# SHIPP — Platform preflight (answers the spec's open questions 1, 2 and 6)

Checks what this workspace actually offers before we build the RAG / Agent / App stages:
- Vector Search (the engine behind "Databricks AI Search") endpoints
- Model Serving endpoints (LLM + vision candidates + embeddings)
- Whether the Bronze CDC history tables expose Delta history (needed for pinned reads)

Nothing is created or changed. **Send a screenshot of every output to your team lead / me** —
the choice of vision model and embedding endpoint depends on it.

In [ ]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
print("Workspace:", w.config.host)
print("Current user:", w.current_user.me().user_name)

## Vector Search endpoints

In [ ]:
try:
    endpoints = list(w.vector_search_endpoints.list_endpoints())
    print(f"{len(endpoints)} Vector Search endpoint(s)")
    for e in endpoints:
        state = e.endpoint_status.state if e.endpoint_status else None
        print(f"  {e.name:<40} type={e.endpoint_type}  state={state}")
except Exception as e:
    print("Vector Search NOT accessible:", type(e).__name__, str(e)[:300])

## Model Serving endpoints (look for an LLM, a vision-capable model, and an embedding model)

In [ ]:
try:
    serving = list(w.serving_endpoints.list())
    print(f"{len(serving)} serving endpoint(s)")
    for s in serving:
        task = getattr(s, "task", None)
        ready = s.state.ready if s.state else None
        print(f"  {s.name:<55} task={task}  ready={ready}")
except Exception as e:
    print("Model Serving NOT accessible:", type(e).__name__, str(e)[:300])

## Bronze CDC tables — Delta history available?

In [ ]:
for t in ["bootcamp_students.shipp_bronze.lb_listings_history",
          "bootcamp_students.shipp_bronze.lb_requests_history"]:
    try:
        v = spark.sql(f"DESCRIBE HISTORY {t} LIMIT 1").first()["version"]
        print(f"  {t}: Delta history OK (latest version {v})")
    except Exception as e:
        print(f"  {t}: NO Delta history ({type(e).__name__}) — pinned reads fall back to live reads")

## Schemas available in the catalog

In [ ]:
display(spark.sql("SHOW SCHEMAS IN bootcamp_students"))